# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YoussefZaky208/Flyrank-ML-Track-Assignemnt/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup — connect to the warehouse

Run this first. It grabs my HF token from Colab Secrets, lists the real files in the dataset,
and loads `dim_clients`, `dim_content`, and just the `month=2026-03` partition of the daily
table (mid-panel month — not `_sample`, since that's the sealed final month).

In [1]:
# Setup: token, dataset id, and safe (non-hardcoded) auth
import os

REPO_ID = "FlyRank/internship-warehouse"
MONTH = "2026-03"  # mid-panel month -- NOT the _sample table (that is the sealed final month)

try:
    from google.colab import userdata  # type: ignore
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("Hugging Face READ token (input hidden, not saved to the notebook): ")

print("Token loaded:", bool(HF_TOKEN))


Token loaded: True


In [2]:
# Discover the real file layout instead of guessing paths
from huggingface_hub import HfApi
import pandas as pd

api = HfApi()
all_files = api.list_repo_files(REPO_ID, repo_type="dataset", token=HF_TOKEN)
print(f"Total files in the dataset repo: {len(all_files)}")

dim_client_files = [f for f in all_files if "dim_clients" in f]
dim_content_files = [f for f in all_files if "dim_content" in f]
daily_month_files = [f for f in all_files if "fact_content_daily_performance/" in f and f"month={MONTH}" in f]

print("dim_clients files:", dim_client_files)
print("dim_content files:", dim_content_files[:5], "..." if len(dim_content_files) > 5 else "")
print(f"fact_content_daily_performance month={MONTH} files:", len(daily_month_files))
daily_month_files[:5]


Total files in the dataset repo: 24
dim_clients files: ['dim_clients.parquet']
dim_content files: ['dim_content.parquet'] 
fact_content_daily_performance month=2026-03 files: 1


['fact_content_daily_performance/month=2026-03/data_0.parquet']

In [3]:
# Load the three tables. Each hf:// read uses the token via storage_options.
def read_hf_parquet(path):
    return pd.read_parquet(f"hf://datasets/{REPO_ID}/{path}", storage_options={"token": HF_TOKEN})

dim_clients = pd.concat([read_hf_parquet(f) for f in dim_client_files], ignore_index=True)
dim_content = pd.concat([read_hf_parquet(f) for f in dim_content_files], ignore_index=True)
daily = pd.concat([read_hf_parquet(f) for f in daily_month_files], ignore_index=True)

print("dim_clients:", dim_clients.shape)
print("dim_content:", dim_content.shape)
print(f"daily (month={MONTH}):", daily.shape)
print()
print("dim_clients columns:", dim_clients.columns.tolist())
print("dim_content columns:", dim_content.columns.tolist())
print("daily columns:", daily.columns.tolist())


dim_clients: (104, 9)
dim_content: (519606, 26)
daily (month=2026-03): (9841378, 31)

dim_clients columns: ['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']
dim_content columns: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']
daily columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pagevie

In [4]:
# Small helper: real warehouse column names may differ slightly from the starter CSV's
# names, so resolve each logical field against whatever actually loaded, instead of guessing.
def first_present(df, candidates, label):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of {candidates} found for '{label}'. Columns available: {list(df.columns)}")

CLIENT_KEY = first_present(daily, ["client_hash_id", "client_id"], "client key")
CONTENT_KEY = first_present(daily, ["content_hash_id", "content_id"], "content key")
DATE_COL = first_present(daily, ["report_date", "date"], "report date")
IMPR_COL = first_present(daily, ["impressions", "gsc_impressions"], "impressions")
CLICK_COL = first_present(daily, ["clicks", "gsc_clicks"], "clicks")
POS_COL = first_present(daily, ["gsc_avg_position", "avg_position"], "position")
GA4_FLAG_COL = first_present(daily, ["ga4_data_available"], "GA4 availability flag")

print("Resolved columns ->")
print(" client key:", CLIENT_KEY)
print(" content key:", CONTENT_KEY)
print(" date:", DATE_COL)
print(" impressions:", IMPR_COL)
print(" clicks:", CLICK_COL)
print(" position:", POS_COL)
print(" ga4 flag:", GA4_FLAG_COL)


Resolved columns ->
 client key: client_hash_id
 content key: content_hash_id
 date: report_date
 impressions: gsc_impressions
 clicks: gsc_clicks
 position: gsc_avg_position
 ga4 flag: ga4_data_available


## 1. Unit of analysis + time window

**Raw table:** one row of `fact_content_daily_performance` = one content item, for one client,
on one day.

**My feature frame (built below):** one row = one content item, aggregated over the whole
`month=2026-03` partition. Same content-level grain I used in W1/W2 on the starter CSV.

**Time window:** `2026-03-01` to `2026-03-31` — a mid-panel month, not the sealed final month.

**Tables:** `fact_content_daily_performance` (month=2026-03 only) for the behavior signal,
`dim_content` for content metadata, `dim_clients` for per-client history coverage.

**Target/proxy:** same proxy pattern as W2 — split the month in half, compare clicks:

```text
is_declining_proxy = (clicks_second_half < clicks_first_half)
```

Still a proxy, not a real future outcome — same honesty caveat as W2.

**Excluded on purpose:** `fact_content_query_90d`. Its 90-day window overlaps the end of any
month I pick and I haven't checked the alignment yet, so I'm leaving it out for now.

## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `content_hash_id`, `client_hash_id` | Context | join/group keys, pseudonyms only |
| `report_date` | Context | defines the window, not a feature itself |
| impressions, clicks, position (this month) | Feature | observed, known by month's end |
| `ga4_data_available` | Feature/filter | tells me if a zero is real or "not tracked yet" |
| `word_count`, content age | Feature | static, known long before the month started |
| `clicks_first_half` / `clicks_second_half` | Label components | these define the label — never features |
| `is_declining_proxy` | Label | what I'd rank pages by |
| `gsc_data_start`, `ga4_data_start` | Context | used to check panel coverage, not modeled |

## 3. Verify with queries, build 5 features, and spring the trap

### 3a. Three verification queries

**Query 1 — grain.** Group by (date, client, content) and check no group has more than one row.

In [5]:
# Query 1: grain probe
grain_check = (
    daily.groupby([DATE_COL, CLIENT_KEY, CONTENT_KEY])
    .size()
    .reset_index(name="n")
)
violations = grain_check[grain_check["n"] > 1]
print(f"Rows checked: {len(grain_check):,}")
print(f"Grain violations (should be 0): {len(violations)}")
violations.head(5)


Rows checked: 9,841,378
Grain violations (should be 0): 0


,report_date,client_hash_id,content_hash_id,n


**Query 2 — row count and date span.** Confirm this really is one month.

In [6]:
# Query 2: row count + date span for this slice
print(f"Rows in month={MONTH} partition: {len(daily):,}")
print(f"Distinct content items: {daily[CONTENT_KEY].nunique():,}")
print(f"Distinct clients: {daily[CLIENT_KEY].nunique():,}")
print(f"Date span: {daily[DATE_COL].min()} -> {daily[DATE_COL].max()}")


Rows in month=2026-03 partition: 9,841,378
Distinct content items: 331,437
Distinct clients: 55
Date span: 2026-03-01 -> 2026-03-31


**Query 3 — availability.** Filter `ga4_data_available IS TRUE` and see how many rows
survive — a zero before a client's GA4 start means "not tracked," not "no engagement."

In [7]:
# Query 3: availability filter, IS TRUE semantics
total_rows = len(daily)
available_rows = daily[daily[GA4_FLAG_COL] == True]  # explicit IS TRUE, not truthy/NaN
n_available = len(available_rows)

print(f"Total rows this month: {total_rows:,}")
print(f"Rows with ga4_data_available IS TRUE: {n_available:,} ({n_available / total_rows * 100:.1f}%)")
print(f"Rows filtered OUT (not yet tracked / GSC-only): {total_rows - n_available:,}")


Total rows this month: 9,841,378
Rows with ga4_data_available IS TRUE: 413,966 (4.2%)
Rows filtered OUT (not yet tracked / GSC-only): 9,427,412


### 3b. Five features (content-level, this month only)

Aggregated up from the daily rows above. Each one is fully known by the time the month ends —
that's the "available when?" line for all five.

In [8]:
# Build the content-level feature frame for month=2026-03

agg = (
    daily.groupby(CONTENT_KEY)
    .agg(
        impressions_month=(IMPR_COL, "sum"),
        clicks_month=(CLICK_COL, "sum"),
        avg_position_month=(POS_COL, "mean"),
        days_with_clicks=(CLICK_COL, lambda s: (s > 0).sum()),
        client_hash_id=(CLIENT_KEY, "first"),
    )
    .reset_index()
    .rename(columns={CONTENT_KEY: "content_hash_id"})
)

# Join static content attributes from dim_content
content_cols = [c for c in ["content_hash_id", "content_id", "word_count", "content_age_days"] if c in dim_content.columns]
dim_content_key = first_present(dim_content, ["content_hash_id", "content_id"], "dim_content key")
agg = agg.merge(
    dim_content[content_cols].rename(columns={dim_content_key: "content_hash_id"}),
    on="content_hash_id",
    how="left",
)

print("Feature frame shape:", agg.shape)
agg.head(5)

# 1. impressions_month     -- available when? Fully summed from the month that has already
#    ended by the time a reviewer opens the queue -- no future information used.
# 2. clicks_month           -- same: a completed-window total, known the moment the month closes.
# 3. avg_position_month     -- an average of daily positions already observed this month --
#    purely descriptive of the past, never of what happens next.
# 4. days_with_clicks       -- a count of days-with-activity within the closed window --
#    known in full once the month ends, same as the totals above.
# 5. word_count (from dim_content) -- a static content attribute set when the page was written
#    or last edited, long before this month even started -- always knowable at any decision point.


Feature frame shape: (331437, 7)


,content_hash_id,impressions_month,clicks_month,avg_position_month,days_with_clicks,client_hash_id,word_count
0,content_000005d4ced12088,86,0,72.854861,0,client_9958f0a7ae1df715,NaN
1,content_00001e488b74b799,0,0,NaN,0,client_625b6439094e23e4,NaN
2,content_00007bd2985b77c3,47,0,5.269565,0,client_73cda7b4e4f265ea,NaN
3,content_00008950670cb6b5,0,0,NaN,0,client_def0955f7a377868,2005.0
4,content_0000a348850eb1fc,0,0,NaN,0,client_3ffa76342f366962,811.0


### 3c. The trap — add one label-derived column on purpose

The label compares clicks in the first half of the month vs. the second half. I add
`clicks_half_delta` (second half minus first half) as a "feature" — the exact number the label
thresholds on at zero — fit a quick model, watch the score jump, then delete it.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
import numpy as np

daily["_dt"] = pd.to_datetime(daily[DATE_COL])
midpoint = daily["_dt"].min() + (daily["_dt"].max() - daily["_dt"].min()) / 2

half_clicks = (
    daily.assign(_half=np.where(daily["_dt"] <= midpoint, "first", "second"))
    .groupby([CONTENT_KEY, "_half"])[CLICK_COL]
    .sum()
    .unstack(fill_value=0)
    .rename(columns={"first": "clicks_first_half", "second": "clicks_second_half"})
    .reset_index()
    .rename(columns={CONTENT_KEY: "content_hash_id"})
)

frame = agg.merge(half_clicks, on="content_hash_id", how="left").fillna(0)
frame["is_declining_proxy"] = (frame["clicks_second_half"] < frame["clicks_first_half"]).astype(int)

honest_features = ["impressions_month", "avg_position_month", "days_with_clicks", "word_count", "content_age_days"]
honest_features = [c for c in honest_features if c in frame.columns]

X_honest = frame[honest_features].fillna(0)
y = frame["is_declining_proxy"]
groups = frame["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X_honest, y, groups))

honest_model = LogisticRegression(max_iter=1000)
honest_model.fit(X_honest.iloc[train_idx], y.iloc[train_idx])
honest_auc = roc_auc_score(y.iloc[test_idx], honest_model.predict_proba(X_honest.iloc[test_idx])[:, 1])
print(f"HONEST AUC (5 features, no leak): {honest_auc:.3f}")

# THE TRAP: add the exact quantity the label thresholds on -- one label-derived column
frame["clicks_half_delta"] = frame["clicks_second_half"] - frame["clicks_first_half"]
leaky_features = honest_features + ["clicks_half_delta"]
X_leak = frame[leaky_features].fillna(0)

leak_model = LogisticRegression(max_iter=1000)
leak_model.fit(X_leak.iloc[train_idx], y.iloc[train_idx])
leak_auc = roc_auc_score(y.iloc[test_idx], leak_model.predict_proba(X_leak.iloc[test_idx])[:, 1])
print(f"LEAKY AUC (label-derived column included): {leak_auc:.3f}  <-- jumps toward 1.0, exactly as the notebook 02 lesson predicts")


HONEST AUC (5 features, no leak): 0.883
LEAKY AUC (label-derived column included): 1.000  <-- jumps toward 1.0, exactly as the notebook 02 lesson predicts


In [10]:
# Delete the leak column and keep the honest score -- this is the number I actually report.
del X_leak, leaky_features, leak_model, leak_auc
frame = frame.drop(columns=["clicks_first_half", "clicks_second_half", "clicks_half_delta"])

print("Kept: the HONEST AUC only ->", round(honest_auc, 3))
print("The leaky column and its score have been removed from anything I report going forward.")


Kept: the HONEST AUC only -> 0.883
The leaky column and its score have been removed from anything I report going forward.


## 4. Data limits

**Limitation: unbalanced panel history.** Clients started tracking at very different times
(per the lane guide, a third of clients have little/no usable history). So this month's slice
isn't an even sample of "typical" behavior — newer clients contribute thinner rows, and any
pattern I find could partly be a client-mix effect, not a universal signal. Checked below.

In [11]:
# Support for the named limitation: how spread out is client tracking history?
start_cols = [c for c in ["gsc_data_start", "ga4_data_start"] if c in dim_clients.columns]
if start_cols:
    for c in start_cols:
        s = pd.to_datetime(dim_clients[c], errors="coerce")
        print(f"{c}: min={s.min()}, max={s.max()}, missing={s.isna().sum()} of {len(s)} clients")
else:
    print("Expected columns not found -- actual dim_clients columns:", dim_clients.columns.tolist())


gsc_data_start: min=2025-01-27 00:00:00, max=2026-06-02 00:00:00, missing=37 of 104 clients
ga4_data_start: min=2025-10-29 00:00:00, max=2026-06-01 00:00:00, missing=53 of 104 clients


## Self-check

Before you submit, confirm each line honestly:

- [x] Five plain-words contract answers given (unit of analysis, tables, window, label/proxy, excluded)
- [x] Exactly three verification queries with outputs visible (grain, count+span, availability with `IS TRUE`)
- [x] Five-feature frame built with an "available when?" line per feature
- [x] Deliberate-leak experiment shown (AUC jump) and the leak column then removed
- [x] One named limitation of this slice (unbalanced panel history), backed by a query
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] Committed to my repo under `work/notebooks/` -- then submit your repo URL on the card. Done.